In [11]:
# local
from src.database import check
from src.database import sql

# standard
import os

# third
import pandas as pd
import requests 

In [ ]:
name = 'Gold Ore'

start = "2026-03-2T00:00:00Z" # UTC time | 0226-02-19 is season 8 start 
end = "2026-03-3T00:00:00Z"

condense = 'true'
has_sold = 'true'

limit = '50'
page = '1'

check.name(name)
# time_check(start, end)

def query_date():
    conn = sql_connect()
    cursor = conn.cursor()
    query = "SELECT MAX(created_at) FROM test"
    cursor.execute(query)
    latest_date = cursor.fetchone()[0]
    conn.close()
    return latest_date

def url_page():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        # f'&item={name}'
        f'&condense={condense}&has_sold={has_sold}'
        f'&limit={limit}&page={page}'
        f'&from={start}&to={end}')
    return url

def query_cursor():
    conn = sql_connect()
    cursor = conn.cursor()
    query = "SELECT MAX(cursor) FROM test"
    cursor.execute(query)
    latest_cursor = str(cursor.fetchone()[0])
    conn.close()
    return latest_cursor

def url_cursor():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        f'&item={name}'
        f'&limit={limit}&page={page}'
        f'&condense={condense}&has_sold={has_sold}'
        f'&from={start}&to={end}'
        f"&cursor={latest_cursor}")
    return url
# req = requests.get(url_page())
# req = requests.get(url_cursor())

In [13]:
# request through pagination
# start = query_date() # for update
with requests.Session() as ses:
    req_body = []
    json = {"pagination": {"count": int(limit)}}
    byte_count = 0    
    while json['pagination']['count'] == 50:
        req = ses.get(url_page())
        json = req.json()
        req_body.extend(json['body'])
        print(f'page {page}', end="\r")
        page = str(int(page) + 1) #sometime req stop comming, rate limited?
        byte_count += len(req.content)
    df = pd.json_normalize(req_body, sep=',')
    print(f'get-requests = {page} \nbytes = {byte_count}')

get-requests = 2 
bytes = 553


In [16]:
json

{'version': '1.0.7',
 'status': 'OK',
 'code': 200,
 'query_time': 0.0297,
 'query_date': '2026-03-17T11:03:56Z',
 'stage': 'production',
 'build': '0.15.130.8295',
 'patch': '110-1',
 'meta': {'method': 'GET',
  'request': 'https://api.darkerdb.com/v1/market?key=meowlin&condense=true&has_sold=true&limit=50&page=1&from=2026-03-1T00%3A00%3A00Z&to=2026-03-2T00%3A00%3A00Z',
  'query': {'key': 'meowlin',
   'condense': 'true',
   'has_sold': 'true',
   'limit': 50,
   'page': 1,
   'from': '2026-03-1T00:00:00Z',
   'to': '2026-03-2T00:00:00Z'},
  'params': []},
 'pagination': {'count': 0, 'limit': 50, 'page': 1},
 'body': []}

In [14]:
df = pd.json_normalize(req_body, sep=',')
df

""


In [15]:
conn = sql.connect()
cursor = conn.cursor()

df_columns = set(df.columns)
dtype_map = {
    "int64": "BIGINT",
    "object": "TEXT", # works for dtype('O')
    "float64": "DOUBLE PRECISION",
    "bool": "BOOLEAN",
    "datetime64[ns]": "TIMESTAMP"
}

# create table if not exists
create_columns = [f'{col} {dtype_map.get(str(df[col].dtype))}' for col in df_columns]
query = f'''
    CREATE TABLE IF NOT EXISTS "season8_solditems" ({','.join(create_columns)},
    PRIMARY KEY (cursor)
    );'''
cursor.execute(query)

# add missing columns
query = """
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = 'season8_solditems'
    ;"""
cursor.execute(query)
sql_cols = cursor.fetchall()
sql_columns = set([column[0] for column in sql_cols])
missing_columns = list(df_columns - sql_columns)
if missing_columns:
    add_columns = [f"ADD COLUMN {col} {dtype_map.get(str(df[col].dtype))}" for col in missing_columns]
    query = f"""ALTER table season8_solditems {','.join(add_columns)};"""
    cursor.execute(query)

# insert data into columns
rows = [tuple(instance.get(c) for c in df.columns) for instance in req_body]
query = f'''
    INSERT INTO "season8_solditems" ({','.join(df.columns)}) 
    VALUES ({','.join(["%s"] * len(df.columns))})
    ON CONFLICT (cursor) DO NOTHING;
    '''
cursor.executemany(query, rows)

conn.commit()
conn.close()

SyntaxError: syntax error at or near ","
LINE 2:     CREATE TABLE IF NOT EXISTS "season8_solditems" (,
                                                            ^
